In [2]:

import torch
import bisect
# import numpy as np
# from numpy.dtypes import StringDType
import csv
from random import randrange
import subprocess
import mmap
import matplotlib.pyplot as plt

In [2]:
#!pip install colorama
from colorama import Fore, Style
#from google.colab import drive
#drive.mount('/content/drive')

In [3]:
#%cd /content/drive/MyDrive/PyTorch/Tokenisation/padunk/model_versions/tokeniser2000_windowsize96_LSTM+GRUdecoder

In [ ]:
batch_size = 32
transformer_window_length = 96
token_vocabulary_size = 8039
transformer_stride = transformer_window_length//2
token_embedding_dim=256
char_vocab_size=38 #this needs to be determined from the input

In [5]:
def countLines (file_path) -> int:
  line_count = 0
  with open(file_path, "r") as python_filehandle:
    with mmap.mmap(python_filehandle.fileno(), length=0, access=mmap.ACCESS_READ) as mmap_filehandle:
        while mmap_filehandle.readline():
            line_count += 1

  return line_count

In [6]:
def retrieveSentence (sentence_idx, tokens_list, sentence_offsets) -> str:
    end_idx = len(tokens_list) if sentence_idx+1 == len(sentence_offsets) else sentence_offsets[sentence_idx+1]
    print(end_idx)
    return "".join(tokens_dict[tokno].replace("<wb>", " ") for tokno in tokens_list[sentence_offsets[sentence_idx]:end_idx]).strip()

In [7]:
def retrieveSubtext(subtext_idx, tokens_list, subtext_offsets) -> str:
    end_idx = len(tokens_list) if subtext_idx+1 == len(subtext_offsets) else subtext_offsets[subtext_idx+1]
    return "".join(tokens_dict[tokno].replace("<wb>", " ") for tokno in tokens_list[subtext_offsets[subtext_idx]:end_idx]).strip()

In [8]:
def retrieveSubtextBeginning(subtext_idx, tokens_list, subtext_offsets) -> str:
    end_idx = subtext_offsets[subtext_idx] + 15
    return "".join(tokens_dict[tokno].replace("<wb>", " ") for tokno in tokens_list[subtext_offsets[subtext_idx]:end_idx]).strip()

In [9]:
def stringifyTokensTensor(tokens_tensor, token_boundaries=False) -> str:
    token_separator = "" if token_boundaries == False else "|"
    return token_separator.join(tokens_dict[tokno].replace("<wb>", " ") for tokno in tokens_tensor.tolist()).strip()

In [13]:
def sortedListFind(sorted_list, sought_after_value) -> int:
    'Locate the leftmost value exactly equal to x'
    i = bisect.bisect_left(sorted_list, sought_after_value)
    if i != len(sorted_list) and sorted_list[i] == sought_after_value:
        return i
    else:
        return -1

In [14]:
def getSubtextWindowIndices(flat_index):
    total = 0
    for s in range(len(subtext_windows)):
        for w in range(subtext_windows[s].size(0)):
            if total == flat_index:
                return s, w
            total += 1

In [22]:
tokens_dict = {}
tokens_dict_reversed = {}
char_dict = {}
char_dict_reversed = {}
with open("../../bpe_token_indices.csv", "r") as bpe_token_indices_file:
    with mmap.mmap(bpe_token_indices_file.fileno(), length=0, access=mmap.ACCESS_READ) as mmap_bpe_file:
        line_no = 0
        for bin_line in iter(mmap_bpe_file.readline, b""):
            line = bin_line.decode("utf-8").strip()
            split_line = line.split(",")
            tokens_dict[int(split_line[0])] = split_line[1].strip()
            tokens_dict_reversed[split_line[1]] = int(split_line[0])
            if line_no < 38:
                char_dict[int(split_line[0])] = split_line[1].strip()
                char_dict_reversed[split_line[1]] = int(split_line[0])
            line_no += 1
            if line_no == token_vocabulary_size:
                break

In [23]:
len(tokens_dict)

8039

In [24]:
def getDatasetInput(tokenised_data_filepath, tagged_data_filepath):

    tokens_list = []
    word_offsets = []
    with open(tokenised_data_filepath, "r") as tokenised_data_file:
        with mmap.mmap(tokenised_data_file.fileno(), length=0, access=mmap.ACCESS_READ) as mmap_tokenised_data_file:
            word_token_count = 0
            for b_line in iter(mmap_tokenised_data_file.readline, b""):
                word_offsets.append(word_token_count)
                for token_no in b_line.decode("utf-8").strip().split(",")[0].split(" "):
                    tokens_list.append(int(token_no))
                    word_token_count += 1
    tokens_tensor = torch.tensor(tokens_list, dtype=torch.int64)

    sentence_offsets = []
    subtext_offsets = []
    text_offsets = []
    row_no = 0
    sentence_no_prev = 0
    subtext_no_prev = 0
    text_id_prev = 0
    token_count = 0
    training_data_wordcount = countLines(tagged_data_filepath) - 1
    pos_tensor = torch.zeros(training_data_wordcount, dtype=torch.int64)
    morph_tag_tensors = []
    for row in csv.DictReader(open(tagged_data_filepath, "r"), delimiter="|"):
        sentence_no = int(row["sentence_no"])
        subtext_no = int(row["subtitle_id"])
        text_id_no = int(row["text_id"])
        if sentence_no != sentence_no_prev:
            sentence_offsets.append(word_offsets[row_no])
            sentence_no_prev = sentence_no
        if text_id_no != text_id_prev:
            text_offsets.append(word_offsets[row_no])
            text_id_prev = text_id_no
            subtext_offsets.append(word_offsets[row_no])
            subtext_no_prev = subtext_no
        elif subtext_no != subtext_no_prev:
            subtext_offsets.append(word_offsets[row_no])
            subtext_no_prev = subtext_no
    
        pos = row["pos"]
        morph_tag = row["morph_tag"]
        pos_tensor[row_no] = pos_dict_reversed[pos]
        morph_tag_tensor = torch.zeros(10, dtype=torch.int64)
        for i in range(10):
            morph_tag_tensor[i] = morph_slots_reverse_dicts[i][morph_tag[i]]
        morph_tag_tensors.append(morph_tag_tensor)        
        row_no += 1
    morph_tag_tensors = torch.stack(morph_tag_tensors, dim=0)

    subtext_windows = []
    flat_subtext_window_tensors = []
    
    subtext_window_loss_masks = []
    flat_token_loss_mask_tensors = []
    flat_window_word_token_lengths = []
    flat_window_word_token_lengths_padded_tensor = []
    pos_window_tensors = []
    morph_tag_window_tensors = []
    
    for i in range(len(subtext_offsets)):
        end_idx = len(tokens_list) if i+1 == len(subtext_offsets) else subtext_offsets[i+1]
        subtext_tokens = tokens_tensor[subtext_offsets[i]:end_idx]
        subtext_token_length = subtext_tokens.size(0)
    
        subtext_chunks = []
        loss_mask_chunks = []
        w = 0
        for j in range(0, subtext_token_length, transformer_stride):
            window_tokens = subtext_tokens[j:j+transformer_window_length]
            padded_window = torch.nn.functional.pad(window_tokens, (0, transformer_window_length-window_tokens.size(0)), value=0)
            loss_mask, start_word_offset, end_word_offset = getWindowLossMask(i, w, padded_window, subtext_offsets, word_offsets, tokens_list)
            if i == 220 and w == 51:
                print(loss_mask, start_word_offset, end_word_offset)
            if start_word_offset != -1:    
                subtext_chunks.append(padded_window)
                flat_subtext_window_tensors.append(padded_window)
                
                loss_mask_chunks.append((loss_mask, start_word_offset, end_word_offset))
                flat_token_loss_mask_tensors.append(loss_mask)
                flat_window_word_token_lengths.append(getWindowWordLengths(start_word_offset, end_word_offset, word_offsets, tokens_list))
                flat_window_word_token_lengths_padded_tensor.append(getWindowWordLengths(start_word_offset, end_word_offset, word_offsets, tokens_list, padded=True))
        
                pos_window, morph_tag_window = getTargetTagWindows(start_word_offset, end_word_offset, pos_tensor, morph_tag_tensors)
                pos_window_tensors.append(pos_window)
                morph_tag_window_tensors.append(morph_tag_window)
    
                w += 1
        subtext_window_loss_masks.append(loss_mask_chunks)       
    
        subtext_windows.append(torch.stack(subtext_chunks, dim=0))
        
    
    flat_token_loss_mask_tensors = torch.stack(flat_token_loss_mask_tensors)
    flat_window_word_token_lengths_padded_tensor = torch.stack(flat_window_word_token_lengths_padded_tensor)
    
    flat_subtext_window_tensors = torch.stack(flat_subtext_window_tensors)
    
    subtext_window_sizes = []
    for subtext_tensors in subtext_windows:
        subtext_window_sizes.append(subtext_tensors.size(0))


    lstm_windows = []
    lstm_word_masks = []
    lstm_word_lengths = []
    lstm_windows_num_words = []
    flat_pos_window_tensors = []
    flat_morph_tag_window_tensors = []
    for s in range(len(subtext_windows)):
        for w in range(subtext_windows[s].size(0)):
            loss_mask, start_word_offset, end_word_offset = subtext_window_loss_masks[s][w]
            lstm_window, lstm_word_mask, lstm_window_word_lengths = getCharLSTMInputWindow(subtext_windows[s][w], loss_mask, start_word_offset, end_word_offset, word_offsets, tokens_list)
            lstm_windows.append(lstm_window)
            lstm_word_masks.append(lstm_word_mask)
            lstm_word_lengths.append(torch.tensor(lstm_window_word_lengths, dtype=torch.int64))
            lstm_windows_num_words.append(len(lstm_window_word_lengths))
    
    lstm_windows = torch.stack(lstm_windows)
    lstm_word_masks = torch.stack(lstm_word_masks)
    lstm_windows_num_words = torch.tensor(lstm_windows_num_words, dtype=torch.int64)

    return flat_subtext_window_tensors, flat_token_loss_mask_tensors, flat_window_word_token_lengths_padded_tensor, lstm_windows, lstm_word_masks, lstm_word_lengths, lstm_windows_num_words, pos_window_tensors, morph_tag_window_tensors

In [52]:
def getInferenceInput(tokenised_data_filepath, headered_csv_text_file):

    tokens_list = []
    word_offsets = []
    with open(tokenised_data_filepath, "r") as tokenised_data_file:
        with mmap.mmap(tokenised_data_file.fileno(), length=0, access=mmap.ACCESS_READ) as mmap_tokenised_data_file:
            word_token_count = 0
            for b_line in iter(mmap_tokenised_data_file.readline, b""):
                word_offsets.append(word_token_count)
                for token_no in b_line.decode("utf-8").strip().split(",")[0].split(" "):
                    tokens_list.append(int(token_no))
                    word_token_count += 1
    tokens_tensor = torch.tensor(tokens_list, dtype=torch.int64)

    sentence_offsets = []
    subtext_offsets = []
    text_offsets = []
    row_no = 0
    sentence_no_prev = 0
    subtext_no_prev = 0
    text_id_prev = 0
    token_count = 0
    #training_data_wordcount = countLines(headered_csv_text_file) - 1
    #pos_tensor = torch.zeros(training_data_wordcount, dtype=torch.int64)
    #morph_tag_tensors = []
    for row in csv.DictReader(open(headered_csv_text_file, "r"), delimiter="|"):
        sentence_no = int(row["sentence_no"])
        subtext_no = int(row["subtitle_id"])
        text_id_no = int(row["text_id"])
        if sentence_no != sentence_no_prev:
            sentence_offsets.append(word_offsets[row_no])
            sentence_no_prev = sentence_no
        if text_id_no != text_id_prev:
            text_offsets.append(word_offsets[row_no])
            text_id_prev = text_id_no
            subtext_offsets.append(word_offsets[row_no])
            subtext_no_prev = subtext_no
        elif subtext_no != subtext_no_prev:
            subtext_offsets.append(word_offsets[row_no])
            subtext_no_prev = subtext_no
    
        #pos = row["pos"]
        #morph_tag = row["morph_tag"]
        #pos_tensor[row_no] = pos_dict_reversed[pos]
        #morph_tag_tensor = torch.zeros(10, dtype=torch.int64)
        #for i in range(10):
            #morph_tag_tensor[i] = morph_slots_reverse_dicts[i][morph_tag[i]]
        #morph_tag_tensors.append(morph_tag_tensor)        
        row_no += 1
    #morph_tag_tensors = torch.stack(morph_tag_tensors, dim=0)

    subtext_windows = []
    flat_subtext_window_tensors = []
    
    subtext_window_loss_masks = []
    flat_token_loss_mask_tensors = []
    flat_window_word_token_lengths = []
    flat_window_word_token_lengths_padded_tensor = []
    #pos_window_tensors = []
    #morph_tag_window_tensors = []
    
    for i in range(len(subtext_offsets)):
        end_idx = len(tokens_list) if i+1 == len(subtext_offsets) else subtext_offsets[i+1]
        subtext_tokens = tokens_tensor[subtext_offsets[i]:end_idx]
        subtext_token_length = subtext_tokens.size(0)
    
        subtext_chunks = []
        loss_mask_chunks = []
        w = 0
        for j in range(0, subtext_token_length, transformer_stride):
            window_tokens = subtext_tokens[j:j+transformer_window_length]
            padded_window = torch.nn.functional.pad(window_tokens, (0, transformer_window_length-window_tokens.size(0)), value=0)
            loss_mask, start_word_offset, end_word_offset = getWindowLossMask(i, w, padded_window, subtext_offsets, word_offsets, tokens_list)
            if start_word_offset != -1:    
                subtext_chunks.append(padded_window)
                flat_subtext_window_tensors.append(padded_window)
                
                loss_mask_chunks.append((loss_mask, start_word_offset, end_word_offset))
                flat_token_loss_mask_tensors.append(loss_mask)
                flat_window_word_token_lengths.append(getWindowWordLengths(start_word_offset, end_word_offset, word_offsets, tokens_list))
                flat_window_word_token_lengths_padded_tensor.append(getWindowWordLengths(start_word_offset, end_word_offset, word_offsets, tokens_list, padded=True))
        
                # pos_window, morph_tag_window = getTargetTagWindows(start_word_offset, end_word_offset, pos_tensor, morph_tag_tensors)
                # pos_window_tensors.append(pos_window)
                # morph_tag_window_tensors.append(morph_tag_window)
    
                w += 1
        subtext_window_loss_masks.append(loss_mask_chunks)       
        #print(i, subtext_chunks)
        subtext_windows.append(torch.stack(subtext_chunks, dim=0))
        
    
    flat_token_loss_mask_tensors = torch.stack(flat_token_loss_mask_tensors)
    flat_window_word_token_lengths_padded_tensor = torch.stack(flat_window_word_token_lengths_padded_tensor)
    
    flat_subtext_window_tensors = torch.stack(flat_subtext_window_tensors)
    
    subtext_window_sizes = []
    for subtext_tensors in subtext_windows:
        subtext_window_sizes.append(subtext_tensors.size(0))


    lstm_windows = []
    lstm_word_masks = []
    lstm_word_lengths = []
    lstm_windows_num_words = []
    # flat_pos_window_tensors = []
    # flat_morph_tag_window_tensors = []
    for s in range(len(subtext_windows)):
        for w in range(subtext_windows[s].size(0)):
            loss_mask, start_word_offset, end_word_offset = subtext_window_loss_masks[s][w]
            lstm_window, lstm_word_mask, lstm_window_word_lengths = getCharLSTMInputWindow(subtext_windows[s][w], loss_mask, start_word_offset, end_word_offset, word_offsets, tokens_list)
            lstm_windows.append(lstm_window)
            lstm_word_masks.append(lstm_word_mask)
            lstm_word_lengths.append(torch.tensor(lstm_window_word_lengths, dtype=torch.int64))
            lstm_windows_num_words.append(len(lstm_window_word_lengths))
    
    lstm_windows = torch.stack(lstm_windows)
    lstm_word_masks = torch.stack(lstm_word_masks)
    lstm_windows_num_words = torch.tensor(lstm_windows_num_words, dtype=torch.int64)

    return flat_subtext_window_tensors, flat_token_loss_mask_tensors, flat_window_word_token_lengths_padded_tensor, lstm_windows, lstm_word_masks, lstm_word_lengths, lstm_windows_num_words

In [26]:
train_dataset_input = getDatasetInput("tokenised_chu_words_training_deepcleaned.csv", "../../chu_words_tagged.csv")

last_whole_word_idx: 76
last_whole_word_idx: 28


In [53]:
assem_inference_input = getInferenceInput("../../tokenised_8000_assem_words_deepcleaned.csv", "../../assem_cyr_full_words.csv")

last_whole_word_idx: 49
last_whole_word_idx: 1


In [28]:
validation_dataset_input = getDatasetInput("tokenised_zogr_validation_words_deepcleaned.csv", "../../zogr_unannotated_validation.csv")

last_whole_word_idx: 55
last_whole_word_idx: 7


In [54]:
assem_inference_input[0].shape, validation_dataset_input[0].shape

(torch.Size([1052, 96]), torch.Size([280, 96]))

In [55]:
stringifyTokensTensor(assem_inference_input[0][20]), assem_inference_input[3][20][:, :10]

('ѣ іродова да събѫдетъ сѧ реченое от га прркмъ глштемъ от егупта възъвахъ снъ моі тогда іродъ відѣвъ ѣко порѫганъ бъіс отъ влъхвъ ізбі всѧ отрокъі сѫштѧѧ въ віѳлеемі і въ всѣхъ прѣдѣлѣхъ еѧ отъ двою лѣтоу і ніже по лѣтоу еже іспъіта отъ влъхвъ тогда събъіс сѧ реченое іереміемь прркмъ глштемъ глас въ рамѣ слъішанъ бъіс плачь і ръіданіе і въпль многь рахіль плачѫшті сѧ чѧдъ своіхъ і не хотѣаше оутѣшіті сѧ ѣко не сѫтъ оумер',
 tensor([[16, 21, 12, 34, 12, 18, 17,  0,  0,  0],
         [34, 17,  0,  0,  0,  0,  0,  0,  0,  0],
         [23,  6, 31, 30, 34, 11, 32,  6,  0,  0],
         [23, 24,  0,  0,  0,  0,  0,  0,  0,  0],
         [21, 11,  8, 11, 10, 12, 11,  0,  0,  0],
         [12, 32,  0,  0,  0,  0,  0,  0,  0,  0],
         [28, 17,  0,  0,  0,  0,  0,  0,  0,  0],
         [37, 21, 21, 13, 14,  6,  0,  0,  0,  0],
         [28, 36,  2, 32, 11, 14,  6,  0,  0,  0],
         [12, 32,  0,  0,  0,  0,  0,  0,  0,  0],
         [11, 28, 35, 37, 32, 17,  0,  0,  0,  0],
         [1

In [45]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [46]:
print(device)

cpu


In [ ]:
class OcsBertModel(torch.nn.Module):
    def __init__(self, token_vocab_size=token_vocabulary_size, token_embedding_dim=token_embedding_dim, token_seq_length=transformer_window_length, attention_heads=4, trans_layers=4, char_vocab_size=38, char_embedding_dim=32, lstm_hidden_size=64, lstm_layers=1, tag_slots_num=11, tag_slot_embedding_dim=32, decoder_lstm_hidden_size=256):
        super().__init__()
       
        self.token_embedder = torch.nn.Embedding(num_embeddings=token_vocab_size, embedding_dim=token_embedding_dim, padding_idx=0)
        self.positional_embedder = torch.nn.Embedding(num_embeddings=token_seq_length, embedding_dim=token_embedding_dim)
        transformer_encoder_layer = torch.nn.TransformerEncoderLayer(d_model=token_embedding_dim, nhead=attention_heads, dim_feedforward=4*token_embedding_dim, batch_first=True)
        self.transformer = torch.nn.TransformerEncoder(transformer_encoder_layer, trans_layers)

        self.token_linear_classifier_head = torch.nn.Linear(token_embedding_dim, token_vocabulary_size)
        
        
        self.token_embedding_dim = token_embedding_dim
        self.token_vocab_size = token_vocab_size
        self.MASK_token_idx = token_vocab_size -1
        self.register_buffer("position_ids", torch.arange(token_seq_length).unsqueeze_(0))

    def corrupt_tokens(self, token_windows: torch.Tensor, padding_mask: torch.Tensor):
        B = token_windows.shape[0]
        W = token_windows.shape[1]
        flat_unpadded_token_windows = token_windows[padding_mask] #(B, W)[(B, W)] -> (N)
        N = flat_unpadded_token_windows.shape[0]

        flat_corrupted_token_windows = flat_unpadded_token_windows.clone()

        flat_corruption_mask = token_windows.new_zeros(N, dtype=torch.bool)
        percent_of_indices = torch.randperm(N, device=token_windows.device)[:int(N*0.15)] #.randperm(N) generates random indices between 0 and N-1, then we select the first 15% of them
        flat_corruption_mask[percent_of_indices] = True

        P = percent_of_indices.shape[0]
        r = torch.rand(P, device=token_windows.device)
        MASK_indices = percent_of_indices[r < 0.8]
        other_token_indices = percent_of_indices[(r >= 0.8) & (r < 0.9)] #has to be bitwise AND because (r < 0.9) produces a tensor 
        #unchanged_indices = percent_of_indices[r >= 0.9]

        flat_corrupted_token_windows[MASK_indices] = 200 #pretend 200 is my token-index for the [MASK] token
        flat_corrupted_token_windows[other_token_indices] = torch.randint(1, self.token_vocab_size, (other_token_indices.shape[0],))

        flat_padding_mask = padding_mask.reshape(B*W)

        flattened_repadded_corrupted_token_windows = flat_corrupted_token_windows.new_zeros(B*W)
        flattened_repadded_corrupted_token_windows[flat_padding_mask] = flat_corrupted_token_windows

        corrupted_token_windows = flattened_repadded_corrupted_token_windows.reshape(B, W)

        return corrupted_token_windows, flat_corruption_mask


    
    def forward(self, token_windows, masked_tokens_mask, token_windows_word_lengths, lstm_windows, lstm_word_masks, lstm_word_lengths, lstm_windows_num_words, flat_pos_window_tensors=None, flat_morph_tag_window_tensors=None) -> torch.Tensor:
        
        corrupted_tokens_windows, flat_corruption_mask = self.corrupt_tokens(token_windows);
        
        ### TRANSFORMER ###
        token_embeddings = self.token_embedder(corrupted_tokens_windows)
        positional_embeddings = self.positional_embedder(self.position_ids.expand(token_embeddings.size(0), -1))
        padding_mask = (token_windows == 0)
        
        token_states = self.transformer(token_embeddings + positional_embeddings, src_key_padding_mask=padding_mask) #(B, W, 256)

        ###MASKED TOKEN PREDICTION###
        flat_corrupted_tokens = token_states[padding_mask][flat_corruption_mask] #(Nmasked, 256)

        corrupted_token_logits = self.token_linear_classifier_head(flat_corrupted_tokens) #(Nmasked, token_vocab_size)

        return corrupted_token_logits

In [ ]:
x = torch.tensor(range(1, 101))
N = x.shape[0]
flat_corruption_mask = torch.zeros(N, dtype=torch.bool)
percent_of_indices = torch.randperm(N)[:int(N*0.15)] #.randperm(N) generates random indices between 0 and N-1, then we select the first 15% of them
flat_corruption_mask[percent_of_indices] = True

P = percent_of_indices.shape[0]
r = torch.rand(P)
MASK_indices = percent_of_indices[r < 0.8]
other_token_indices = percent_of_indices[(r >= 0.8) & (r < 0.9)] #if r>= 0.8 produces [False, False, True, False] and r < 0.9 produces [True, False, True, True], then the first element-wise bitwise ANDed with the second produces [False, False, True, False] 
unchanged_indices = percent_of_indices[r >= 0.9]

print(percent_of_indices, MASK_indices, other_token_indices, unchanged_indices)
x[MASK_indices] = 200 #pretend 200 is my token-index for the [MASK] token
x[other_token_indices] = torch.randint(1, token_vocabulary_size, (other_token_indices.shape[0],))
x, flat_corruption_mask, x[flat_corruption_mask]

tensor([61, 15, 45, 36, 12, 39, 86, 55, 67, 54, 77, 10, 42, 53, 20]) tensor([15, 45, 12, 39, 86, 55, 54, 10, 53, 20]) tensor([67]) tensor([61, 36, 77, 42])


(tensor([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10, 200,  12, 200,  14,
          15, 200,  17,  18,  19,  20, 200,  22,  23,  24,  25,  26,  27,  28,
          29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39, 200,  41,  42,
          43,  44,  45, 200,  47,  48,  49,  50,  51,  52,  53, 200, 200, 200,
          57,  58,  59,  60,  61,  62,  63,  64,  65,  66,  67,   9,  69,  70,
          71,  72,  73,  74,  75,  76,  77,  78,  79,  80,  81,  82,  83,  84,
          85,  86, 200,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,  98,
          99, 100]),
 tensor([False, False, False, False, False, False, False, False, False, False,
          True, False,  True, False, False,  True, False, False, False, False,
          True, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False,  True, False, False,  True,
         False, False,  True, False, False,  True, False, False, False, False,
         False, False, False,  

In [48]:
class textWindowsDataset(torch.utils.data.Dataset):
    def __init__(self, token_windows, token_window_loss_masks, token_window_word_lengths, lstm_windows, lstm_word_masks, lstm_word_lengths, lstm_windows_num_words, pos_window_tensors, morph_tag_window_tensors):
        self.token_windows = token_windows
        self.token_window_loss_masks = token_window_loss_masks
        self.token_window_word_lengths = token_window_word_lengths
        self.lstm_windows = lstm_windows
        self.lstm_word_masks = lstm_word_masks
        self.lstm_word_lengths = lstm_word_lengths
        self.lstm_windows_num_words = lstm_windows_num_words
        self.pos_window_tensors = pos_window_tensors
        self.morph_tag_window_tensors = morph_tag_window_tensors

    def __len__(self):
        return len(self.token_windows)

    def __getitem__(self, idx):
        
        return {'token_windows': self.token_windows[idx], 'token_window_loss_masks': self.token_window_loss_masks[idx], 'token_window_word_lengths': self.token_window_word_lengths[idx], 'lstm_windows': self.lstm_windows[idx], 'lstm_word_masks': self.lstm_word_masks[idx], 'lstm_word_lengths': self.lstm_word_lengths[idx], 'lstm_windows_num_words': self.lstm_windows_num_words[idx], 'pos_window_tensors': self.pos_window_tensors[idx], 'morph_tag_window_tensors': self.morph_tag_window_tensors[idx]}

In [56]:
class inferenceWindowsDataset(torch.utils.data.Dataset):
    def __init__(self, token_windows, token_window_loss_masks, token_window_word_lengths, lstm_windows, lstm_word_masks, lstm_word_lengths, lstm_windows_num_words):
        self.token_windows = token_windows
        self.token_window_loss_masks = token_window_loss_masks
        self.token_window_word_lengths = token_window_word_lengths
        self.lstm_windows = lstm_windows
        self.lstm_word_masks = lstm_word_masks
        self.lstm_word_lengths = lstm_word_lengths
        self.lstm_windows_num_words = lstm_windows_num_words
        # self.pos_window_tensors = pos_window_tensors
        # self.morph_tag_window_tensors = morph_tag_window_tensors

    def __len__(self):
        return len(self.token_windows)

    def __getitem__(self, idx):
        
        return {'token_windows': self.token_windows[idx], 'token_window_loss_masks': self.token_window_loss_masks[idx], 'token_window_word_lengths': self.token_window_word_lengths[idx], 'lstm_windows': self.lstm_windows[idx], 'lstm_word_masks': self.lstm_word_masks[idx], 'lstm_word_lengths': self.lstm_word_lengths[idx], 'lstm_windows_num_words': self.lstm_windows_num_words[idx]}

In [49]:
def data_loader_collate_fn(samples):
    return {
        "token_windows" : torch.stack([sample["token_windows"] for sample in samples]),
        "token_window_loss_masks" : torch.stack([sample["token_window_loss_masks"] for sample in samples]),
        "token_window_word_lengths" : torch.stack([sample["token_window_word_lengths"] for sample in samples]),
        "lstm_windows" : torch.stack([sample["lstm_windows"] for sample in samples]),
        "lstm_word_masks" : torch.stack([sample["lstm_word_masks"] for sample in samples]),
        "lstm_word_lengths" : torch.cat([sample["lstm_word_lengths"] for sample in samples]),
        "lstm_windows_num_words" : torch.stack([sample["lstm_windows_num_words"] for sample in samples]),
        "pos_window_tensors" : torch.cat([sample["pos_window_tensors"] for sample in samples]),
        "morph_tag_window_tensors" : torch.cat([sample["morph_tag_window_tensors"] for sample in samples])      
    }

In [51]:
def inference_data_loader_collate_fn(samples):
    return {
        "token_windows" : torch.stack([sample["token_windows"] for sample in samples]),
        "token_window_loss_masks" : torch.stack([sample["token_window_loss_masks"] for sample in samples]),
        "token_window_word_lengths" : torch.stack([sample["token_window_word_lengths"] for sample in samples]),
        "lstm_windows" : torch.stack([sample["lstm_windows"] for sample in samples]),
        "lstm_word_masks" : torch.stack([sample["lstm_word_masks"] for sample in samples]),
        "lstm_word_lengths" : torch.cat([sample["lstm_word_lengths"] for sample in samples]),
        "lstm_windows_num_words" : torch.stack([sample["lstm_windows_num_words"] for sample in samples])
    }

In [81]:
# mydataset = textWindowsDataset(flat_subtext_window_tensors, flat_token_loss_mask_tensors, flat_window_word_token_lengths_padded_tensor, lstm_windows, lstm_word_masks, lstm_word_lengths, pos_window_tensors, morph_tag_window_tensors)

train_dataset = textWindowsDataset(*train_dataset_input)

# def data_loader_collate_fn(samples):
#     return samples

train_data_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=data_loader_collate_fn)

validation_dataset = textWindowsDataset(*validation_dataset_input)

validation_data_loader = torch.utils.data.DataLoader(validation_dataset, batch_size=32, shuffle=False, collate_fn=data_loader_collate_fn)

assem_inference_dataset = inferenceWindowsDataset(*assem_inference_input)
assem_inference_data_loader = torch.utils.data.DataLoader(assem_inference_dataset, batch_size=32, shuffle=False, collate_fn=inference_data_loader_collate_fn)

In [67]:
network = MorphologyLSTMTransformerModel()
cross_entropy_loss = torch.nn.modules.loss.CrossEntropyLoss()
adamw_optimiser = torch.optim.AdamW(network.parameters(), lr=1e-4)
network.to(device)
#network.load_state_dict(torch.load("MorphologyLSTMTransformerModel_POS_only_18epochs.pt", map_location=device))

MorphologyLSTMTransformerModel(
  (token_embedder): Embedding(8039, 256, padding_idx=0)
  (positional_embedder): Embedding(96, 256)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=1024, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=1024, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (char_embedder): Embedding(38, 32, padding_idx=0)
  (char_dropout): Dropout(p=0.1, inplace=False)
  (lstm): LSTM(32, 64, batch_first=

In [ ]:
train_data_loader_iter = iter(train_data_loader)
network.train()

In [ ]:
try:
    batch = next(train_data_loader_iter)
except StopIteration:
    print("Reached end of dataset")    
tkn_wndw = batch['token_windows'].to(device)
tkn_msks = batch['token_window_loss_masks'].to(device)
tkn_lngths = batch['token_window_word_lengths'].to(device)
lstm_wndws = batch['lstm_windows'].to(device)
lstm_msks = batch['lstm_word_masks'].to(device)
lstm_lngths = batch['lstm_word_lengths'].to(device) #this is a flattened list of the lengths of all the full words in the batch
lstm_wndws_num_wrds = batch['lstm_windows_num_words'].to(device) #this is a flattened list of the number of words in each window in the batch
flat_pos_window_tensors = batch['pos_window_tensors'].to(device)
flat_morph_tag_tensors = batch['morph_tag_window_tensors'].to(device)

#print(lstm_wndws_num_wrds)


output_logits = network(tkn_wndw, tkn_msks, tkn_lngths, lstm_wndws, lstm_msks, lstm_lngths, lstm_wndws_num_wrds, flat_pos_window_tensors, flat_morph_tag_tensors)

losses = [cross_entropy_loss(output_logits[0], flat_pos_window_tensors)]
for i in range(1, 11):
    losses.append(cross_entropy_loss(output_logits[i], flat_morph_tag_tensors[:, i -1]))
loss = torch.stack(losses).mean()
adamw_optimiser.zero_grad()
loss.backward()
adamw_optimiser.step()
print(loss)

In [ ]:
#torch.save(obj=network.state_dict(), f="decoder_lstm_1epoch.pt")
network.load_state_dict(torch.load("google_20epochs_lstm_decoder.pt", map_location=device))

In [ ]:
network.eval()
validation_dataloader_iter = iter(validation_data_loader)


In [ ]:
batched_output_logits[0][active_windows_masks_for_each_word_pos.T].shape, flat_pos_window_tensors.shape

In [ ]:
training_loss_values = []
batches_counts = []
training_epochs = []
per_epoch_loss_values = []
per_epoch_tagslot_loss_values = []

In [ ]:
validation_average_per_epoch_loss_values = []
validation_per_epoch_tagslot_loss_values = []

In [ ]:
network.train()

batches_count = 0
best_validation_loss = 99999.9
for epoch_no in range(30):
    train_data_loader_iter = iter(train_data_loader)
    network.train()

    epoch_loss_values = []
    epoch_tagslot_loss_values = []
    for batch in train_data_loader_iter:
        tkn_wndw = batch['token_windows'].to(device)
        tkn_msks = batch['token_window_loss_masks'].to(device)
        tkn_lngths = batch['token_window_word_lengths'].to(device)
        lstm_wndws = batch['lstm_windows'].to(device)
        lstm_msks = batch['lstm_word_masks'].to(device)
        lstm_lngths = batch['lstm_word_lengths'].to(device)
        lstm_wndws_num_wrds = batch['lstm_windows_num_words'].to(device)
        flat_pos_window_tensors = batch['pos_window_tensors'].to(device)
        flat_morph_tag_tensors = batch['morph_tag_window_tensors'].to(device)
    
        output_logits = network(tkn_wndw, tkn_msks, tkn_lngths, lstm_wndws, lstm_msks, lstm_lngths, lstm_wndws_num_wrds, flat_pos_window_tensors, flat_morph_tag_tensors)
        
        losses = [cross_entropy_loss(output_logits[0], flat_pos_window_tensors)]
        for i in range(1, 11):
            losses.append(cross_entropy_loss(output_logits[i], flat_morph_tag_tensors[:, i -1]))
        loss = torch.stack(losses).mean()
        adamw_optimiser.zero_grad()
        loss.backward()
        adamw_optimiser.step()

        
        training_loss_values.append(loss.item())
        epoch_loss_values.append(loss.detach())
        epoch_tagslot_loss_values.append(torch.stack(losses).detach())
        
        batches_counts.append(batches_count)
        batches_count += 1
    
    epoch_training_loss = torch.stack(epoch_loss_values).mean().item()
    per_epoch_loss_values.append(epoch_training_loss)
    per_epoch_tagslot_loss_values.append(torch.stack(epoch_tagslot_loss_values, dim=0).mean(dim=0).detach().cpu())

    # if len(per_epoch_loss_values) > 5 and per_epoch_loss_values[-1] < per_epoch_loss_values[-2]:
    #     torch.save(obj=network.state_dict(), f="MorphologyLSTMTransformerModel_bestcheckpoint.pt")

    network.eval()
    validation_dataloader_iter = iter(validation_data_loader)
    validation_per_window_loss_values = []
    validation_per_window_tagslot_loss_values = []
    with torch.inference_mode():
        for batch in validation_dataloader_iter:
            tkn_wndw = batch['token_windows'].to(device)
            tkn_msks = batch['token_window_loss_masks'].to(device)
            tkn_lngths = batch['token_window_word_lengths'].to(device)
            lstm_wndws = batch['lstm_windows'].to(device)
            lstm_msks = batch['lstm_word_masks'].to(device)
            lstm_lngths = batch['lstm_word_lengths'].to(device)
            lstm_wndws_num_wrds = batch['lstm_windows_num_words'].to(device)
            flat_pos_window_tensors = batch['pos_window_tensors'].to(device)
            flat_morph_tag_tensors = batch['morph_tag_window_tensors'].to(device)
        
            batched_output_logits, batched_predicted_idcs, active_windows_masks_for_each_word_pos = network(tkn_wndw, tkn_msks, tkn_lngths, lstm_wndws, lstm_msks, lstm_lngths, lstm_wndws_num_wrds, flat_pos_window_tensors, flat_morph_tag_tensors) #[(B, Wmax, slot_possibilities)*11], (B, Wmax, 11), (Wmax, B)
        
            losses = [cross_entropy_loss(batched_output_logits[0][active_windows_masks_for_each_word_pos.T], flat_pos_window_tensors)]
            for i in range(1, 11):
                losses.append(cross_entropy_loss(batched_output_logits[i][active_windows_masks_for_each_word_pos.T], flat_morph_tag_tensors[:, i-1]))
            loss = torch.stack(losses).mean()
            # loss = losses[0] #test only part-of-speech training
            validation_per_window_loss_values.append(loss)
            validation_per_window_tagslot_loss_values.append(torch.stack(losses)) #11
        mean_validation_per_batch_loss = torch.stack(validation_per_window_loss_values).mean().item()
        validation_average_per_epoch_loss_values.append(mean_validation_per_batch_loss)
    
        #print(validation_per_window_tagslot_loss_values)
        
        mean_validation_per_batch_tagslot_loss = torch.stack(validation_per_window_tagslot_loss_values, dim=0).mean(dim=0).detach().cpu()
        validation_per_epoch_tagslot_loss_values.append(mean_validation_per_batch_tagslot_loss)

        if epoch_no > 9:
          print(f"Epoch no. {epoch_no}\n    Training loss: {epoch_training_loss}\n    Validation loss: {mean_validation_per_batch_loss}")
          if mean_validation_per_batch_loss < best_validation_loss:
            torch.save(obj=network.state_dict(), f="lstm_decoder_tokeniser_8000_windowsize_96_best_checkpoint.pt")
            print(f"Saving model at epoch no. {epoch_no} because its validation loss was lower than the previous best validation-loss of {best_validation_loss}")
            best_validation_loss = mean_validation_per_batch_loss
    training_epochs.append(epoch_no)
torch.save(obj=network.state_dict(), f=f"tokeniser_8000_30_epochs.pt")

In [ ]:
network.train()

for param in network.transformer.parameters():
    param.requires_grad = False
for param in network.token_embedder.parameters():
    param.requires_grad = False
for param in network.positional_embedder.parameters():
    param.requires_grad = False

batches_count = batches_counts[-1]
for epoch_no in range(15, 30):
    train_data_loader_iter = iter(train_data_loader)
    network.train()
    network.transformer.eval()

    epoch_loss_values = []
    epoch_tagslot_loss_values = []
    for batch in train_data_loader_iter:
        tkn_wndw = batch['token_windows'].to(device)
        tkn_msks = batch['token_window_loss_masks'].to(device)
        tkn_lngths = batch['token_window_word_lengths'].to(device)
        lstm_wndws = batch['lstm_windows'].to(device)
        lstm_msks = batch['lstm_word_masks'].to(device)
        lstm_lngths = batch['lstm_word_lengths'].to(device)
        flat_pos_window_tensors = batch['pos_window_tensors'].to(device)
        flat_morph_tag_tensors = batch['morph_tag_window_tensors'].to(device)
    
        output_logits = network(tkn_wndw, tkn_msks, tkn_lngths, lstm_wndws, lstm_msks, lstm_lngths, flat_pos_window_tensors, flat_morph_tag_tensors)
        
        losses = [cross_entropy_loss(output_logits[0], flat_pos_window_tensors)]
        for i in range(1, 11):
            losses.append(cross_entropy_loss(output_logits[i], flat_morph_tag_tensors[:, i -1]))
        weighted_losses = losses.copy()
        weighted_losses[0] = 4*weighted_losses[0]
        loss = torch.stack(weighted_losses).mean()
        # loss = losses[0] #test only part-of-speech training
        adamw_optimiser.zero_grad()
        loss.backward()
        adamw_optimiser.step()

        actual_loss = torch.stack(losses).mean()
        training_loss_values.append(actual_loss.item())
        epoch_loss_values.append(actual_loss.detach())
        epoch_tagslot_loss_values.append(torch.stack(losses).detach())
        
        batches_counts.append(batches_count)
        batches_count += 1
    
    per_epoch_loss_values.append(torch.stack(epoch_loss_values).mean().item())
    per_epoch_tagslot_loss_values.append(torch.stack(epoch_tagslot_loss_values, dim=0).mean(dim=0).detach().cpu())

    # if len(per_epoch_loss_values) > 5 and per_epoch_loss_values[-1] < per_epoch_loss_values[-2]:
    #     torch.save(obj=network.state_dict(), f="MorphologyLSTMTransformerModel_bestcheckpoint.pt")

    network.eval()
    validation_dataloader_iter = iter(validation_data_loader)
    validation_per_window_loss_values = []
    validation_per_window_tagslot_loss_values = []
    for batch in validation_dataloader_iter:
        tkn_wndw = batch['token_windows'].to(device)
        tkn_msks = batch['token_window_loss_masks'].to(device)
        tkn_lngths = batch['token_window_word_lengths'].to(device)
        lstm_wndws = batch['lstm_windows'].to(device)
        lstm_msks = batch['lstm_word_masks'].to(device)
        lstm_lngths = batch['lstm_word_lengths'].to(device)
        flat_pos_window_tensors = batch['pos_window_tensors'].to(device)
        flat_morph_tag_tensors = batch['morph_tag_window_tensors'].to(device)
    
        lgts, idcs = network(tkn_wndw, tkn_msks, tkn_lngths, lstm_wndws, lstm_msks, lstm_lngths, flat_pos_window_tensors, flat_morph_tag_tensors)
    
        losses = [cross_entropy_loss(lgts[0], flat_pos_window_tensors)]
        for i in range(1, 11):
            losses.append(cross_entropy_loss(lgts[i], flat_morph_tag_tensors[:, i -1]))
        loss = torch.stack(losses).mean()
        # loss = losses[0] #test only part-of-speech training
        validation_per_window_loss_values.append(loss)
        validation_per_window_tagslot_loss_values.append(torch.stack(losses)) #11
    mean_validation_per_batch_loss = torch.stack(validation_per_window_loss_values).mean().item()
    validation_average_per_epoch_loss_values.append(mean_validation_per_batch_loss)

    #print(validation_per_window_tagslot_loss_values)
    
    mean_validation_per_batch_tagslot_loss = torch.stack(validation_per_window_tagslot_loss_values, dim=0).mean(dim=0).detach().cpu()
    validation_per_epoch_tagslot_loss_values.append(mean_validation_per_batch_tagslot_loss)

    training_epochs.append(epoch_no)
    

In [ ]:
validation_per_epoch_tagslot_loss_values, validation_average_per_epoch_loss_values, per_epoch_loss_values, per_epoch_tagslot_loss_values

In [ ]:
plt.plot(batches_counts, training_loss_values, label="training loss")
plt.ylabel("Loss")
plt.xlabel("batches processed")
plt.legend()
print(training_loss_values[batches_counts[-1]])
plt.show()

In [ ]:
per_epoch_loss_values, per_epoch_tagslot_loss_values, validation_average_per_epoch_loss_values

In [ ]:
plt.plot(training_epochs, validation_average_per_epoch_loss_values)

In [ ]:
plt.plot(training_epochs, per_epoch_loss_values)

In [61]:
!pwd

/home/joe/torot_2023/PyTorch/Tokenisation/padunk/model_versions/tokeniser8000_windowsize96_LSTM+GRUdecoder_POS_only


In [93]:
network.load_state_dict(torch.load("tokeniser_8000_overfitted_29_epochs.pt", map_location=device))

<All keys matched successfully>

In [ ]:
#torch.save(obj=network.state_dict(), f="MorphologyLSTMTransformerModel_15epochs_joint_15epochs_LSTM_only.pt")

In [94]:
network.eval()
validation_dataloader_iter = iter(validation_data_loader)

In [95]:
try:
    batch = next(validation_dataloader_iter)
except StopIteration:
    print("Reached end of dataset")
tkn_wndw = batch['token_windows'].to(device)
tkn_msks = batch['token_window_loss_masks'].to(device)
tkn_lngths = batch['token_window_word_lengths'].to(device)
lstm_wndws = batch['lstm_windows'].to(device)
lstm_msks = batch['lstm_word_masks'].to(device)
lstm_lngths = batch['lstm_word_lengths'].to(device)
lstm_wndws_num_wrds = batch['lstm_windows_num_words'].to(device)
flat_pos_window_tensors = batch['pos_window_tensors'].to(device)
flat_morph_tag_tensors = batch['morph_tag_window_tensors'].to(device)

with torch.inference_mode():
    batched_output_logits, batched_predicted_idcs, active_windows_masks_for_each_word_pos = network(tkn_wndw, tkn_msks, tkn_lngths, lstm_wndws, lstm_msks, lstm_lngths, lstm_wndws_num_wrds, flat_pos_window_tensors, flat_morph_tag_tensors) #[(B, Wmax, slot_possibilities)*11], (B, Wmax, 11), (Wmax, B)

losses = [cross_entropy_loss(batched_output_logits[0][active_windows_masks_for_each_word_pos.T], flat_pos_window_tensors)]
for i in range(1, 11):
    losses.append(cross_entropy_loss(batched_output_logits[i][active_windows_masks_for_each_word_pos.T], flat_morph_tag_tensors[:, i-1]))
loss = torch.stack(losses).mean()
print(loss)

idcs = batched_predicted_idcs[active_windows_masks_for_each_word_pos.T]

tensor(0.1540)


In [ ]:
idcs.shape

In [ ]:
lstm_wndws[lstm_msks][1]


stringifyLSTMWindowEntry(lstm_wndws[lstm_msks][2001])

In [ ]:
lstm_wndws[lstm_msks].shape[0:2]

In [ ]:
#stringifyPosTensor(flat_pos_window_tensors[1300:1400]), stringifyPosTensor(idcs[0][1300:1400])



for i in range(700,800):
    actual_pos = stringifyPosTensor(flat_pos_window_tensors[i:i+1])
    predicted_pos = stringifyPosTensor(idcs[i:i+1, 0])
    actual_morph_tag = stringifyMorphTagsTensor(flat_morph_tag_tensors[i:i+1])
    predicted_morph_tag = stringifyMorphTagsTensor(idcs[i:i+1, 1:])
    word = stringifyLSTMWindowEntry(lstm_wndws[lstm_msks][i])
    
    if actual_pos != predicted_pos:
        predicted_pos = Fore.RED+predicted_pos+Style.RESET_ALL
    if actual_morph_tag != predicted_morph_tag:
        predicted_morph_tag = Fore.RED+predicted_morph_tag+Style.RESET_ALL
    
    print(actual_pos, predicted_pos, actual_morph_tag, predicted_morph_tag, word)

In [103]:
network.eval()
assem_iter = iter(assem_inference_data_loader)

In [104]:
try:
    batch = next(assem_iter)
except StopIteration:
    print("Reached end of dataset")
tkn_wndw = batch['token_windows'].to(device)
tkn_msks = batch['token_window_loss_masks'].to(device)
tkn_lngths = batch['token_window_word_lengths'].to(device)
lstm_wndws = batch['lstm_windows'].to(device)
lstm_msks = batch['lstm_word_masks'].to(device)
lstm_lngths = batch['lstm_word_lengths'].to(device)
lstm_wndws_num_wrds = batch['lstm_windows_num_words'].to(device)

with torch.inference_mode():
    batched_output_logits, batched_predicted_idcs, active_windows_masks_for_each_word_pos = network(tkn_wndw, tkn_msks, tkn_lngths, lstm_wndws, lstm_msks, lstm_lngths, lstm_wndws_num_wrds) #[(B, Wmax, slot_possibilities)*11], (B, Wmax, 11), (Wmax, B)

    idcs = batched_predicted_idcs[active_windows_masks_for_each_word_pos.T]

In [107]:
#stringifyPosTensor(flat_pos_window_tensors[1300:1400]), stringifyPosTensor(idcs[0][1300:1400])



for i in range(600,700):
    #actual_pos = stringifyPosTensor(flat_pos_window_tensors[i:i+1])
    predicted_pos = stringifyPosTensor(idcs[i:i+1, 0])
    #actual_morph_tag = stringifyMorphTagsTensor(flat_morph_tag_tensors[i:i+1])
    predicted_morph_tag = stringifyMorphTagsTensor(idcs[i:i+1, 1:])
    word = stringifyLSTMWindowEntry(lstm_wndws[lstm_msks][i])
    
    # if actual_pos != predicted_pos:
    #     predicted_pos = Fore.RED+predicted_pos+Style.RESET_ALL
    # if actual_morph_tag != predicted_morph_tag:
    #     predicted_morph_tag = Fore.RED+predicted_morph_tag+Style.RESET_ALL
    
    print(predicted_pos, predicted_morph_tag, word)

V- 3saia----i бъіс
G- ---------n да
V- 3spia----i събѫдет
Pk 3s---qa--i сѧ
V- -suppnn-wi реченое
R- ---------n отъ
Nb -s---mg--i га
Ne -s---ma--i ісаіем
Nb -s---ma--i прркмъ
V- -pppamn-si глштемъ
I- ---------n се
Nb -s---fn--i дѣва
R- ---------n въ
Nb -s---nl--i чрѣвѣ
V- 3spia----i прііметъ
C- ---------n і
V- 3spia----i родітъ
Nb -s---mg--i сна
C- ---------n і
V- 3ppia----i нарекѫтъ
Nb -s---na--i імѧ
Pp 3s---md--i емоу
Ne -s---mn--i енманоуіль
Pr -s---nn--i еже
V- 3spia----i естъ
V- -spppnn-si ськазаемо
R- ---------n сь
Pp 1p---qi--i намі
Nb -s---mn--i бъ
V- -supamn-si въставъ
Df ---------n же
Ne -s---mn--i іосіфь
R- ---------n отъ
Ne -s---mg--i сьна
V- 3saia----i сьтворі
Dq ---------n ѣкоже
V- 3saia----i повелѣ
Pp 3s---md--i емоу
Nb -s---mn--i англъ
C- ---------n і
V- 3saia----i пріѧтъ
Nb -s---fa--i женѫ
Pt 3s---fa--i своѭ
Nb -s---mg--i га
Ne -s---ma--i ісаіем
Nb -s---ma--i прркмъ
V- -pppamn-si глштемъ
Pd -s---na--i се
Nb -s---mg--i дѣва
R- ---------n въ
Nb -s---nl--i чрѣвѣ
V- 3spia--

In [ ]:
#just crap to understand the loss-functions
loss = torch.nn.CrossEntropyLoss()
sft_mx = torch.nn.Softmax(dim=0)
inp = torch.tensor([-1.5, 1.4, 0.1, 3.2], requires_grad=True)
print(inp)
print(sft_mx(inp))
targ = torch.empty(1, dtype=torch.long).random_(4)
print(targ)
output = loss(inp, targ)
print(output)